In [0]:
dbutils.widgets.remove("xml_path")

In [0]:
dbutils.library.restartPython()

In [0]:

import json
import traceback

# ============================================================
# 1. XML PATH WIDGET
# ============================================================

dbutils.widgets.text(
    "xml_path",
    "/Volumes/idmc_poc/bronze/idmc_files/landing/IDMC_Medium_Complexity_Mapping_Updated.xml",
    "IDMC XML Path"
)

xml_path = dbutils.widgets.get("xml_path")

print("=" * 90)
print("IDMC TO DATABRICKS - COMPLETE MIGRATION ENGINE")
print("=" * 90)

print("XML Path:", xml_path)


# ============================================================
# 2. IMPORT MIGRATION PIPELINE AND HUMAN GATEWAY
# ============================================================

from pipeline.migration_pipeline import MigrationPipeline
from Human_GateWay.human_gateway import HumanGateway

print("\nMigrationPipeline imported successfully!")
print("HumanGateway imported successfully!")


# ============================================================
# 3. DATABRICKS AUTHENTICATION
# ============================================================

DATABRICKS_TOKEN = (
    dbutils.notebook
    .entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

if not DATABRICKS_TOKEN:
    raise ValueError("Databricks authentication token is unavailable.")


# ============================================================
# 4. WORKSPACE CONFIGURATION
# ============================================================

workspace_url = (
    "https://dbc-ffb099c2-fea8.cloud.databricks.com"
)

LLM_MODEL = "databricks-gpt-oss-120b"

OUTPUT_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark"
)

REVIEW_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark/human_reviews"
)

print("\nWorkspace URL:", workspace_url)
print("LLM Model:", LLM_MODEL)
print("Output Directory:", OUTPUT_DIR)


# ============================================================
# 5. APPROVED HUMAN GATEWAY REVIEW IDS
# ============================================================

APPROVED_REVIEWS = {
    "m_customer_standardize":
        "f5f2499d-d3c9-4475-bed5-3cf7cf2085ff",

    "m_order_enrichment":
        "299d9780-71e1-4845-b685-9a2f05c5994f",

    "m_inventory_reconciliation":
        "80f32658-8b80-4279-b844-002bb4195c9f",

    "m_daily_sales_summary":
        "c72222ce-55cd-4341-8982-2574a5b9aac7"
}

gateway = HumanGateway(review_dir=REVIEW_DIR)

print("\n" + "=" * 90)
print("VERIFYING HUMAN GATEWAY APPROVALS")
print("=" * 90)

for mapping_name, review_id in APPROVED_REVIEWS.items():

    review = gateway.get_review(review_id)

    if not review:
        raise ValueError(
            f"Review not found for mapping: {mapping_name}"
        )

    if review.get("mapping_name") != mapping_name:
        raise ValueError(
            f"Review mapping mismatch: {mapping_name}"
        )

    if review.get("status") != "PASS":
        raise ValueError(
            f"Review is not approved: {mapping_name}"
        )

    if not review.get("validations_passed"):
        raise ValueError(
            f"Review validations failed: {mapping_name}"
        )

    print(
        f"{mapping_name} | "
        f"Review: APPROVED | "
        f"Review ID: {review_id}"
    )

print("\nAll four Human Gateway approvals verified!")


# ============================================================
# 6. SAFE MIGRATION CONFIGURATION
# ============================================================

DRY_RUN = True
EXECUTION_APPROVED = False
ALLOW_SYNTHETIC_MERGE_KEYS = False

print("\n" + "=" * 90)
print("MIGRATION EXECUTION CONFIGURATION")
print("=" * 90)

print("Dry Run:", DRY_RUN)
print("Execution Approved:", EXECUTION_APPROVED)
print("Synthetic Merge Keys Allowed:", ALLOW_SYNTHETIC_MERGE_KEYS)

print("Target writes are disabled.")


# ============================================================
# 7. RUN ALL FOUR MAPPINGS INDIVIDUALLY
# ============================================================

results = []

print("\n" + "=" * 90)
print("STARTING FINAL APPROVED MIGRATION VALIDATION")
print("=" * 90)

for index, (mapping_name, review_id) in enumerate(
    APPROVED_REVIEWS.items(),
    start=1
):

    print("\n" + "#" * 90)
    print(f"MAPPING {index}/4: {mapping_name}")
    print("#" * 90)

    try:

        pipeline = MigrationPipeline(

            xml_path=xml_path,

            spark=spark,

            output_dir=OUTPUT_DIR,

            workspace_url=workspace_url,

            databricks_token=DATABRICKS_TOKEN,

            llm_model=LLM_MODEL,

            # Run only the current mapping
            selected_mapping_name=mapping_name,

            # Safe execution configuration
            dry_run=DRY_RUN,

            execution_approved=EXECUTION_APPROVED,

            allow_synthetic_merge_keys=ALLOW_SYNTHETIC_MERGE_KEYS,

            verbose=True,

            human_review_dir=REVIEW_DIR,

            # Mapping-specific approved review
            approved_review_id=review_id

        )

        mapping_result = pipeline.run()

        # Normalize result format
        if isinstance(mapping_result, list):

            if len(mapping_result) == 0:
                raise RuntimeError(
                    f"No result returned for {mapping_name}"
                )

            for item in mapping_result:
                results.append(item)

        elif isinstance(mapping_result, dict):

            results.append(mapping_result)

        else:

            raise TypeError(
                f"Unexpected result type for {mapping_name}: "
                f"{type(mapping_result)}"
            )

    except Exception as error:

        print("\nMAPPING EXECUTION ERROR:")
        traceback.print_exc()

        results.append({
            "mapping_name": mapping_name,
            "success": False,
            "stage": "PIPELINE_EXCEPTION",
            "executed": False,
            "error": str(error)
        })


# ============================================================
# 8. INDIVIDUAL MAPPING RESULTS
# ============================================================

print("\n" + "=" * 90)
print("INDIVIDUAL MAPPING RESULTS")
print("=" * 90)

for index, result in enumerate(results, start=1):

    print(f"\nMAPPING RESULT {index}")
    print("-" * 90)

    print(
        json.dumps(
            result,
            indent=4,
            ensure_ascii=False,
            default=str
        )
    )


# ============================================================
# 9. FINAL MIGRATION ENGINE SUMMARY
# ============================================================

successful = [
    result for result in results
    if result.get("success") is True
]

failed = [
    result for result in results
    if result.get("success") is not True
]

print("\n" + "=" * 90)
print("FINAL MIGRATION ENGINE SUMMARY")
print("=" * 90)

print("Expected mappings       :", len(APPROVED_REVIEWS))
print("Total results returned  :", len(results))
print("Successful mappings     :", len(successful))
print("Failed / blocked        :", len(failed))

print("\nMAPPING STATUS")

for result in results:

    print(
        f"{result.get('mapping_name', 'UNKNOWN')} | "
        f"Success: {result.get('success')} | "
        f"Stage: {result.get('stage')} | "
        f"Executed: {result.get('executed')}"
    )


# ============================================================
# 10. FAILURE DETAILS
# ============================================================

if failed:

    print("\n" + "=" * 90)
    print("FAILED / BLOCKED MAPPINGS")
    print("=" * 90)

    for result in failed:

        print("\nMapping:", result.get("mapping_name"))
        print("Stage:", result.get("stage"))
        print("Error:", result.get("error"))
        print("Review ID:", result.get("review_id"))


# ============================================================
# 11. FINAL COMPLETION STATUS
# ============================================================

print("\n" + "=" * 90)
print("MIGRATION ENGINE VALIDATION COMPLETED")
print("=" * 90)

all_passed = (
    len(results) == len(APPROVED_REVIEWS)
    and len(successful) == len(APPROVED_REVIEWS)
)

if all_passed:

    print("STATUS: ALL FOUR MAPPINGS PASSED")
    print("STATUS: HUMAN GATEWAY APPROVALS VERIFIED")
    print("STATUS: GENERATED PYSPARK VALIDATION COMPLETED")

else:

    print("STATUS: ENGINE VALIDATION NEEDS ATTENTION")

print("\nDry Run:", DRY_RUN)
print("Execution Approved:", EXECUTION_APPROVED)
print("Target Writes Performed: NO")

print("=" * 90)


In [0]:
# ============================================================
# IDMC TO DATABRICKS
# FINAL LIVE MIGRATION EXECUTION - FOUR MAPPINGS
# ============================================================

import json
import traceback

from pipeline.migration_pipeline import MigrationPipeline
from Human_GateWay.human_gateway import HumanGateway


# ============================================================
# 1. CONFIGURATION
# ============================================================

XML_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/landing/"
    "IDMC_Medium_Complexity_Mapping_Updated.xml"
)

OUTPUT_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark"
)

REVIEW_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark/human_reviews"
)

WORKSPACE_URL = (
    "https://dbc-ffb099c2-fea8.cloud.databricks.com"
)

LLM_MODEL = "databricks-gpt-oss-120b"


# ============================================================
# 2. APPROVED HUMAN GATEWAY REVIEW IDS
# ============================================================

APPROVED_REVIEWS = {
    "m_customer_standardize":
        "f5f2499d-d3c9-4475-bed5-3cf7cf2085ff",

    "m_order_enrichment":
        "299d9780-71e1-4845-b685-9a2f05c5994f",

    "m_inventory_reconciliation":
        "80f32658-8b80-4279-b844-002bb4195c9f",

    "m_daily_sales_summary":
        "c72222ce-55cd-4341-8982-2574a5b9aac7"
}


# ============================================================
# 3. SELECT MAPPINGS
# ============================================================

# Customer standardization has already executed successfully.
# Therefore, execute only the remaining three mappings.

MAPPINGS_TO_RUN = [
    "m_order_enrichment",
    "m_inventory_reconciliation",
    "m_daily_sales_summary"
]

# To intentionally rerun all four mappings, replace the list above with:
#
# MAPPINGS_TO_RUN = list(APPROVED_REVIEWS.keys())
#
# MERGE will update matching records in existing target tables.


# ============================================================
# 4. DATABRICKS AUTHENTICATION
# ============================================================

DATABRICKS_TOKEN = (
    dbutils.notebook
    .entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

if not DATABRICKS_TOKEN:
    raise RuntimeError(
        "Databricks authentication token unavailable."
    )


# ============================================================
# 5. INITIALIZE HUMAN GATEWAY
# ============================================================

gateway = HumanGateway(review_dir=REVIEW_DIR)

print("=" * 100)
print("VERIFYING HUMAN GATEWAY APPROVALS")
print("=" * 100)


# ============================================================
# 6. VERIFY APPROVALS BEFORE EXECUTION
# ============================================================

for mapping_name in MAPPINGS_TO_RUN:

    review_id = APPROVED_REVIEWS[mapping_name]

    review = gateway.get_review(review_id)

    if not review:
        raise RuntimeError(
            f"Human Gateway review not found: {mapping_name}"
        )

    decision_data = review.get("decision")

    # The decision may be stored as a dictionary.
    if isinstance(decision_data, dict):
        decision_value = decision_data.get("decision")
    else:
        decision_value = decision_data

    if review.get("mapping_name") != mapping_name:
        raise RuntimeError(
            f"Review mapping mismatch: {mapping_name}"
        )

    if review.get("status") != "PASS":
        raise RuntimeError(
            f"Review status is not PASS: {mapping_name}"
        )

    if decision_value != "APPROVE":
        raise RuntimeError(
            f"Review decision is not APPROVE: {mapping_name}"
        )

    if review.get("validations_passed") is not True:
        raise RuntimeError(
            f"Review validations did not pass: {mapping_name}"
        )

    if (
        "can_execute" in review
        and review.get("can_execute") is not True
    ):
        raise RuntimeError(
            f"Execution permission denied: {mapping_name}"
        )

    print(
        f"{mapping_name} | "
        f"APPROVED | "
        f"Review ID: {review_id}"
    )

print("\nAll selected mapping approvals verified.")


# ============================================================
# 7. LIVE EXECUTION CONFIGURATION
# ============================================================

DRY_RUN = False
EXECUTION_APPROVED = True
ALLOW_SYNTHETIC_MERGE_KEYS = True

print("\n" + "=" * 100)
print("LIVE MIGRATION EXECUTION CONFIGURATION")
print("=" * 100)

print("Dry Run:", DRY_RUN)
print("Execution Approved:", EXECUTION_APPROVED)
print(
    "Synthetic POC Merge Keys:",
    ALLOW_SYNTHETIC_MERGE_KEYS
)

print("Mappings selected:", MAPPINGS_TO_RUN)


# ============================================================
# 8. EXECUTE EACH MAPPING INDIVIDUALLY
# ============================================================

results = []

for index, mapping_name in enumerate(
    MAPPINGS_TO_RUN,
    start=1
):

    review_id = APPROVED_REVIEWS[mapping_name]

    print("\n" + "#" * 100)
    print(
        f"STARTING MAPPING {index}/{len(MAPPINGS_TO_RUN)}: "
        f"{mapping_name}"
    )
    print("#" * 100)

    try:

        pipeline = MigrationPipeline(

            xml_path=XML_PATH,

            spark=spark,

            output_dir=OUTPUT_DIR,

            workspace_url=WORKSPACE_URL,

            databricks_token=DATABRICKS_TOKEN,

            llm_model=LLM_MODEL,

            selected_mapping_name=mapping_name,

            # LIVE EXECUTION
            dry_run=DRY_RUN,

            execution_approved=EXECUTION_APPROVED,

            allow_synthetic_merge_keys=(
                ALLOW_SYNTHETIC_MERGE_KEYS
            ),

            verbose=True,

            human_review_dir=REVIEW_DIR,

            approved_review_id=review_id
        )

        mapping_result = pipeline.run()

        if isinstance(mapping_result, list):

            if not mapping_result:
                raise RuntimeError(
                    f"No result returned: {mapping_name}"
                )

            results.extend(mapping_result)

        elif isinstance(mapping_result, dict):

            results.append(mapping_result)

        else:

            raise TypeError(
                f"Unexpected pipeline result type: "
                f"{type(mapping_result)}"
            )

    except Exception as error:

        print("\nMAPPING EXECUTION FAILED")

        traceback.print_exc()

        results.append({
            "mapping_name": mapping_name,
            "success": False,
            "stage": "PIPELINE_EXCEPTION",
            "executed": False,
            "error": str(error)
        })


# ============================================================
# 9. INDIVIDUAL EXECUTION RESULTS
# ============================================================

print("\n" + "=" * 100)
print("INDIVIDUAL MIGRATION RESULTS")
print("=" * 100)

for result in results:

    print("\n")

    print(
        json.dumps(
            result,
            indent=4,
            ensure_ascii=False,
            default=str
        )
    )


# ============================================================
# 10. FINAL MIGRATION SUMMARY
# ============================================================

successful = [
    result for result in results
    if result.get("success") is True
    and result.get("executed") is True
]

failed = [
    result for result in results
    if not (
        result.get("success") is True
        and result.get("executed") is True
    )
]

print("\n" + "=" * 100)
print("FINAL LIVE MIGRATION SUMMARY")
print("=" * 100)

print("Expected selected mappings:", len(MAPPINGS_TO_RUN))
print("Total results returned:", len(results))
print("Successfully executed:", len(successful))
print("Failed / blocked:", len(failed))

print("\nMAPPING STATUS")

for result in results:

    print(
        f"{result.get('mapping_name')} | "
        f"Success: {result.get('success')} | "
        f"Executed: {result.get('executed')} | "
        f"Stage: {result.get('stage')} | "
        f"Validation: {result.get('validation_status')}"
    )


# ============================================================
# 11. FAILURE DETAILS
# ============================================================

if failed:

    print("\nFAILED MAPPINGS")

    for result in failed:

        print("\nMapping:", result.get("mapping_name"))
        print("Stage:", result.get("stage"))
        print("Error:", result.get("error"))


# ============================================================
# 12. COMPLETION STATUS
# ============================================================

all_successful = (
    len(results) == len(MAPPINGS_TO_RUN)
    and len(successful) == len(MAPPINGS_TO_RUN)
)

print("\n" + "=" * 100)

if all_successful:

    print("STATUS: ALL SELECTED MAPPINGS EXECUTED SUCCESSFULLY")

else:

    print("STATUS: MIGRATION EXECUTION NEEDS ATTENTION")

print("=" * 100)

In [0]:
# ============================================================
# FINAL POST-MIGRATION VALIDATION - ALL FOUR TARGETS
# ============================================================

from pyspark.sql import functions as F

TARGETS = {
    "m_customer_standardize": {
        "table": "idmc_poc.migration_test.mig4_customer_standardized",
        "keys": ["customer_id"]
    },
    "m_order_enrichment": {
        "table": "idmc_poc.migration_test.mig4_order_enriched",
        "keys": ["order_id"]
    },
    "m_inventory_reconciliation": {
        "table": "idmc_poc.migration_test.mig4_inventory_position",
        "keys": ["product_id", "warehouse_id"]
    },
    "m_daily_sales_summary": {
        "table": "idmc_poc.migration_test.mig4_daily_sales_summary",
        "keys": ["order_date"]
    }
}

final_results = []

for mapping, config in TARGETS.items():

    table = config["table"]
    keys = config["keys"]

    df = spark.table(table)

    null_condition = F.lit(False)

    for key in keys:
        null_condition = null_condition | F.col(key).isNull()

    null_rows = df.filter(null_condition).count()

    duplicate_groups = (
        df.groupBy(*keys)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    result = {
        "mapping": mapping,
        "target": table,
        "row_count": df.count(),
        "column_count": len(df.columns),
        "null_key_rows": null_rows,
        "duplicate_key_groups": duplicate_groups,
        "status": (
            "PASS"
            if null_rows == 0 and duplicate_groups == 0
            else "REVIEW_REQUIRED"
        )
    }

    final_results.append(result)

    print(result)

print("\nFINAL VALIDATION:",
      "ALL PASS" if all(
          x["status"] == "PASS" for x in final_results
      ) else "REVIEW REQUIRED")

In [0]:
# ============================================================
# FINAL POST-MIGRATION VALIDATION - ALL FOUR TARGETS
# ============================================================

from pyspark.sql import functions as F

TARGETS = {
    "m_customer_standardize": {
        "table": "idmc_poc.migration_test.mig4_customer_standardized",
        "keys": ["customer_id"]
    },
    "m_order_enrichment": {
        "table": "idmc_poc.migration_test.mig4_order_enriched",
        "keys": ["order_id"]
    },
    "m_inventory_reconciliation": {
        "table": "idmc_poc.migration_test.mig4_inventory_position",
        "keys": ["product_id", "warehouse_id"]
    },
    "m_daily_sales_summary": {
        "table": "idmc_poc.migration_test.mig4_daily_sales_summary",
        "keys": ["order_date"]
    }
}

final_results = []

for mapping, config in TARGETS.items():

    table = config["table"]
    keys = config["keys"]

    df = spark.table(table)

    null_condition = F.lit(False)

    for key in keys:
        null_condition = null_condition | F.col(key).isNull()

    null_rows = df.filter(null_condition).count()

    duplicate_groups = (
        df.groupBy(*keys)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    result = {
        "mapping": mapping,
        "target": table,
        "row_count": df.count(),
        "column_count": len(df.columns),
        "null_key_rows": null_rows,
        "duplicate_key_groups": duplicate_groups,
        "status": (
            "PASS"
            if null_rows == 0 and duplicate_groups == 0
            else "REVIEW_REQUIRED"
        )
    }

    final_results.append(result)

    print(result)

print("\nFINAL VALIDATION:",
      "ALL PASS" if all(
          x["status"] == "PASS" for x in final_results
      ) else "REVIEW REQUIRED")

In [0]:

# ============================================================
# DISPLAY GENERATED PYSPARK CODE FOR ALL MAPPINGS
# ============================================================

GENERATED_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark"
)

MAPPINGS = [
    "m_customer_standardize",
    "m_order_enrichment",
    "m_inventory_reconciliation",
    "m_daily_sales_summary"
]

for mapping_name in MAPPINGS:

    file_path = f"{GENERATED_DIR}/{mapping_name}.py"

    print("\n" + "=" * 100)
    print(f"GENERATED PYSPARK: {mapping_name}")
    print("=" * 100)

    try:
        code = dbutils.fs.head(file_path, 1000000)

        print(code)

        print("\nFile:", file_path)
        print("Status: Successfully read")

    except Exception as e:
        print("Unable to read generated code:", str(e))


In [0]:
parsed_path = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "parsed_metadata/idmc_parsed_objects.json"
)

parsed_df = spark.read.option("multiline", "true").json(parsed_path)

display(parsed_df)

In [0]:
# ============================================================
# VIEW GENERATED PYSPARK CODE
# ============================================================

import os

mapping_name = "m_customer_standardize"

generated_file_path = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "generated_pyspark/"
    f"{mapping_name}.py"
)

if os.path.exists(generated_file_path):

    with open(generated_file_path, "r", encoding="utf-8") as file:
        generated_code = file.read()

    print("=" * 80)
    print(f"GENERATED PYSPARK CODE: {mapping_name}")
    print("=" * 80)

    print(generated_code)

else:
    print("Generated PySpark file not found!")
    print("Expected path:", generated_file_path)

In [0]:
%sql
SELECT *
FROM idmc_poc.migration_test.mig4_order_enriched
LIMIT 20;

In [0]:
%sql
SELECT * FROM idmc_poc.migration_test.src_orders LIMIT 10;

In [0]:
%sql
DESCRIBE TABLE idmc_poc.bronze.orders;

In [0]:
%sql
SHOW TABLES IN idmc_poc.bronze;

In [0]:
%sql
DESCRIBE TABLE idmc_poc.silver.customer;

In [0]:
%sql
SELECT *
FROM idmc_poc.silver.customer
WHERE customer_id IN ('C001', 'C002', 'C003', 'C999');

In [0]:
%sql
DESCRIBE TABLE idmc_poc.migration_test.src_customer_silver;

In [0]:
%sql
SELECT *
FROM idmc_poc.migration_test.src_customer_silver
ORDER BY customer_id;

In [0]:
%sql
DESCRIBE TABLE idmc_poc.bronze.customer;

In [0]:
%sql
DESCRIBE TABLE idmc_poc.silver.customer;

In [0]:

from datetime import datetime
from decimal import Decimal

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DecimalType, TimestampType
)

spark.sql("CREATE SCHEMA IF NOT EXISTS idmc_poc.migration_test")

SCHEMA = "idmc_poc.migration_test"


def create_delta_table(table_name, schema, rows):
    full_name = f"{SCHEMA}.{table_name}"

    df = spark.createDataFrame(rows, schema=schema)

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )

    print(f"Created {full_name}: {df.count()} rows")
    df.printSchema()


# ---------------------------------------------------------
# 1. Customer standardization source
# XML fields: customer_id, first_name, last_name, email,
#             updated_ts
# ---------------------------------------------------------

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("updated_ts", TimestampType(), True),
])

customer_rows = [
    ("C001", "Asha", "Reddy", " ASHA@EXAMPLE.COM ",
     datetime(2026, 10, 20, 9, 0, 0)),
    ("C002", "Ravi", "Kumar", "ravi@example.com",
     datetime(2026, 10, 20, 10, 0, 0)),
    ("C003", "Maya", "Shah", None,
     datetime(2026, 10, 21, 11, 0, 0)),
    (None, "Invalid", "Customer", "invalid@example.com",
     datetime(2026, 10, 21, 12, 0, 0)),
]

create_delta_table(
    "src_customer_standardize",
    customer_schema,
    customer_rows
)


# ---------------------------------------------------------
# 2. Orders source
# XML fields: order_id, customer_id, order_amount, order_ts
# Used by both order enrichment and daily sales.
# ---------------------------------------------------------

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_amount", DecimalType(12, 2), True),
    StructField("order_ts", TimestampType(), True),
])

orders_rows = [
    ("O1001", "C001", Decimal("120.50"),
     datetime(2026, 10, 20, 10, 30, 0)),
    ("O1002", "C002", Decimal("75.00"),
     datetime(2026, 10, 20, 14, 15, 0)),
    ("O1003", "C001", Decimal("200.00"),
     datetime(2026, 10, 21, 9, 45, 0)),
    ("O1004", "C003", Decimal("0.00"),
     datetime(2026, 10, 21, 12, 0, 0)),
    ("O1005", "C999", Decimal("35.00"),
     datetime(2026, 10, 21, 15, 0, 0)),
    ("O1006", "C002", Decimal("-10.00"),
     datetime(2026, 10, 21, 16, 0, 0)),
    (None, "C001", Decimal("50.00"),
     datetime(2026, 10, 21, 17, 0, 0)),
]

create_delta_table(
    "src_orders",
    orders_schema,
    orders_rows
)


# ---------------------------------------------------------
# 3. Customer enrichment source
# XML fields: customer_id, full_name, email_clean
# ---------------------------------------------------------

customer_silver_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("email_clean", StringType(), True),
])

customer_silver_rows = [
    ("C001", "Asha Reddy", "asha@example.com"),
    ("C002", "Ravi Kumar", "ravi@example.com"),
    ("C003", "Maya Shah", None),
]

create_delta_table(
    "src_customer_silver",
    customer_silver_schema,
    customer_silver_rows
)


# ---------------------------------------------------------
# 4. Inventory source
# XML fields: product_id, warehouse_id, on_hand_qty,
#             reserved_qty, updated_ts
# ---------------------------------------------------------

inventory_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("warehouse_id", StringType(), True),
    StructField("on_hand_qty", IntegerType(), True),
    StructField("reserved_qty", IntegerType(), True),
    StructField("updated_ts", TimestampType(), True),
])

inventory_rows = [
    ("P001", "WH01", 100, 20,
     datetime(2026, 10, 20, 8, 0, 0)),
    ("P002", "WH01", 50, 10,
     datetime(2026, 10, 20, 9, 0, 0)),
    ("P001", "WH02", 30, 5,
     datetime(2026, 10, 20, 10, 0, 0)),
    ("P003", "WH01", 10, 15,
     datetime(2026, 10, 21, 8, 0, 0)),
]

create_delta_table(
    "src_inventory",
    inventory_schema,
    inventory_rows
)


# ---------------------------------------------------------
# 5. Product reference source
# XML fields: product_id, product_name, category
# ---------------------------------------------------------

products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
])

products_rows = [
    ("P001", "Laptop", "Electronics"),
    ("P002", "Desk", "Furniture"),
    ("P003", "Chair", "Furniture"),
    ("P004", "Monitor", "Electronics"),
]

create_delta_table(
    "src_products",
    products_schema,
    products_rows
)

print("\nAll five synthetic source tables are ready.")


In [0]:
source_tables = [
    "src_customer_standardize",
    "src_orders",
    "src_customer_silver",
    "src_inventory",
    "src_products",
]

for table in source_tables:
    full_name = f"idmc_poc.migration_test.{table}"

    print(f"\n{full_name}")
    print("Rows:", spark.table(full_name).count())
    display(spark.table(full_name))

In [0]:
# ============================================================
# IDMC -> DATABRICKS POC
# P1: Recreate synthetic source tables + seed sample data
# MAP101–MAP108
# ============================================================

from datetime import datetime, date
from decimal import Decimal
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, DoubleType, DecimalType, TimestampType, DateType
)

CAT = "idmc_poc"

S = StringType()
I = IntegerType()
B = BooleanType()
D = DoubleType()
M = DecimalType(18, 2)
T = TimestampType()
DT = DateType()


def recreate(schema_name, table_name, fields, rows):
    """Drop/recreate one synthetic POC source table and insert sample rows."""
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CAT}`.`{schema_name}`")

    full_name = f"`{CAT}`.`{schema_name}`.`{table_name}`"
    spark.sql(f"DROP TABLE IF EXISTS {full_name}")

    schema = StructType([
        StructField(name, data_type, True)
        for name, data_type in fields
    ])

    df = spark.createDataFrame(rows, schema=schema)
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)

    count = spark.table(f"{CAT}.{schema_name}.{table_name}").count()
    print(f"CREATED {CAT}.{schema_name}.{table_name}: {count} rows")


# ------------------------------------------------------------
# MAP101 - Customer enrichment source
# ------------------------------------------------------------
recreate("silver", "customer", [
    ("customer_id", S),
    ("customer_name", S),
    ("segment", S),
    ("country", S)
], [
    ("C001", "Asha Rao", "GOLD", "IN"),
    ("C002", "Ravi Kumar", "STANDARD", "IN"),
    ("C003", "Mia Chen", "GOLD", "SG"),
    ("C004", "Noah Smith", "STANDARD", "US")
])


# ------------------------------------------------------------
# MAP102 - Product inventory reconciliation
# ------------------------------------------------------------
recreate("erp", "products", [
    ("product_id", S),
    ("sku", S),
    ("product_name", S),
    ("category_id", S)
], [
    ("P001", "SKU-001", "Laptop", "CAT-ELEC"),
    ("P002", "SKU-002", "Keyboard", "CAT-ELEC"),
    ("P003", "SKU-003", "Desk", "CAT-FURN")
])

recreate("wms", "inventory", [
    ("product_id", S),
    ("on_hand_qty", I),
    ("reserved_qty", I),
    ("warehouse_id", S)
], [
    ("P001", 20, 3, "WH-01"),
    ("P002", 50, 10, "WH-01"),
    ("P003", 8, 2, "WH-02"),
    ("P999", 2, 4, "WH-02")  # Intentional data-quality issue
])

recreate("reference", "product_category", [
    ("category_id", S),
    ("category_name", S)
], [
    ("CAT-ELEC", "Electronics"),
    ("CAT-FURN", "Furniture")
])


# ------------------------------------------------------------
# MAP103 - Customer consent and privacy compliance
# ------------------------------------------------------------
recreate("landing", "customer_consent", [
    ("customer_id", S),
    ("consent_type", S),
    ("consent_value", B),
    ("event_ts", T)
], [
    ("C001", "MARKETING", True, datetime(2026, 10, 18, 10, 0)),
    ("C001", "MARKETING", False, datetime(2026, 10, 19, 9, 0)),
    ("C002", "ANALYTICS", True, datetime(2026, 10, 18, 12, 0)),
    ("C003", "MARKETING", False, datetime(2026, 10, 19, 8, 30))
])

recreate("reference", "privacy_policy", [
    ("consent_type", S),
    ("retention_days", I),
    ("policy_version", S)
], [
    ("MARKETING", 365, "v1"),
    ("ANALYTICS", 730, "v2")
])


# ------------------------------------------------------------
# MAP104 - Payment risk
# ------------------------------------------------------------
recreate("payments", "transaction", [
    ("payment_id", S),
    ("customer_id", S),
    ("amount", M),
    ("payment_ts", T),
    ("channel", S)
], [
    ("PAY001", "C001", Decimal("120.00"), datetime(2026, 10, 18, 11), "WEB"),
    ("PAY002", "C002", Decimal("55.50"), datetime(2026, 10, 18, 12), "MOBILE"),
    ("PAY003", "C003", Decimal("999.99"), datetime(2026, 10, 19, 8), "WEB"),
    ("PAY004", "C004", Decimal("1000001.00"), datetime(2026, 10, 19, 9), "API")
])

recreate("risk", "chargebacks", [
    ("payment_id", S),
    ("chargeback_flag", B),
    ("reason_code", S)
], [
    ("PAY001", False, None),
    ("PAY003", True, "CB01")
])

recreate("risk", "device_trust", [
    ("customer_id", S),
    ("device_score", D)
], [
    ("C001", 95.0),
    ("C002", 72.5),
    ("C003", 30.0),
    ("C004", 84.0)
])


# ------------------------------------------------------------
# MAP105 - Web activity and campaign analytics
# ------------------------------------------------------------
recreate("landing", "web_events", [
    ("event_id", S),
    ("visitor_id", S),
    ("event_ts", T),
    ("event_type", S),
    ("page_url", S)
], [
    ("EV001", "V001", datetime(2026, 10, 19, 9, 0), "PAGE_VIEW", "/home"),
    ("EV002", "V001", datetime(2026, 10, 19, 9, 5), "CLICK", "/products"),
    ("EV003", "V001", datetime(2026, 10, 19, 10, 0), "PURCHASE", "/checkout"),
    ("EV004", "V002", datetime(2026, 10, 19, 9, 15), "PAGE_VIEW", "/offers"),
    ("EV005", "V002", datetime(2026, 10, 19, 9, 20), "CLICK", "/offers/sale")
])

recreate("marketing", "campaigns", [
    ("campaign_id", S),
    ("source", S),
    ("medium", S)
], [
    ("CMP001", "search", "cpc"),
    ("CMP002", "newsletter", "email")
])


# ------------------------------------------------------------
# MAP106 - Finance and credit notes
# ------------------------------------------------------------
recreate("erp", "invoice", [
    ("invoice_id", S),
    ("customer_id", S),
    ("net_amount", M),
    ("tax_amount", M),
    ("invoice_date", DT)
], [
    ("INV001", "C001", Decimal("100.00"), Decimal("10.00"), date(2026, 10, 17)),
    ("INV002", "C002", Decimal("200.00"), Decimal("20.00"), date(2026, 10, 18)),
    ("INV003", "C003", Decimal("50.00"), Decimal("5.00"), date(2026, 10, 19))
])

recreate("erp", "credit_note", [
    ("credit_note_id", S),
    ("invoice_id", S),
    ("credit_amount", M)
], [
    ("CN001", "INV002", Decimal("20.00")),
    ("CN002", "INV003", Decimal("5.00"))
])


# ------------------------------------------------------------
# MAP107 - Supplier performance
# ------------------------------------------------------------
recreate("erp", "purchase_orders", [
    ("po_id", S),
    ("supplier_id", S),
    ("ordered_qty", I),
    ("received_qty", I),
    ("delivery_ts", T)
], [
    ("PO001", "SUP001", 100, 100, datetime(2026, 10, 10, 9)),
    ("PO002", "SUP001", 50, 45, datetime(2026, 10, 15, 9)),
    ("PO003", "SUP002", 80, 80, datetime(2026, 10, 16, 9))
])

recreate("erp", "suppliers", [
    ("supplier_id", S),
    ("supplier_name", S),
    ("region", S)
], [
    ("SUP001", "Northstar Supply", "APAC"),
    ("SUP002", "Contoso Parts", "EMEA")
])


# ------------------------------------------------------------
# MAP108 - Support cases and SLA policies
# ------------------------------------------------------------
recreate("crm", "support_cases", [
    ("case_id", S),
    ("customer_id", S),
    ("priority", S),
    ("created_ts", T),
    ("resolved_ts", T)
], [
    ("CASE001", "C001", "HIGH", datetime(2026, 10, 18, 8), datetime(2026, 10, 18, 10)),
    ("CASE002", "C002", "MEDIUM", datetime(2026, 10, 18, 9), datetime(2026, 10, 19, 12)),
    ("CASE003", "C003", "LOW", datetime(2026, 10, 19, 8), None),
    # Intentional DQ failure: resolution precedes creation
    ("CASE004", "C004", "HIGH", datetime(2026, 10, 19, 7), datetime(2026, 10, 19, 6))
])

recreate("reference", "sla_policy", [
    ("priority", S),
    ("sla_hours", I)
], [
    ("HIGH", 4),
    ("MEDIUM", 24),
    ("LOW", 72)
])


# ------------------------------------------------------------
# MAP101 - File source for orders_delta
# Spark CSV output is a directory containing CSV part file(s).
# ------------------------------------------------------------
orders_path = (
    "/Volumes/idmc_poc/bronze/idmc_files/landing/orders_delta.csv"
)

dbutils.fs.rm(orders_path, True)

orders_schema = StructType([
    StructField("order_id", S, True),
    StructField("customer_id", S, True),
    StructField("order_status", S, True),
    StructField("amount", M, True),
    StructField("updated_ts", T, True)
])

orders_rows = [
    ("ORD001", "C001", "COMPLETED", Decimal("120.00"), datetime(2026, 10, 18, 10)),
    ("ORD002", "C002", "PENDING", Decimal("0.00"), datetime(2026, 10, 18, 11)),
    ("ORD003", "C003", "COMPLETED", Decimal("75.25"), datetime(2026, 10, 19, 8)),
    # Intentional DQ failures: negative amount and unknown customer
    ("ORD004", "C999", "INVALID", Decimal("-5.00"), datetime(2026, 10, 19, 9))
]

orders_df = spark.createDataFrame(orders_rows, schema=orders_schema)

orders_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(orders_path)

print(f"Created CSV source at {orders_path}: {orders_df.count()} rows")


# ------------------------------------------------------------
# Validate table existence, row counts and schemas
# ------------------------------------------------------------
tables = [
    "silver.customer",
    "erp.products",
    "wms.inventory",
    "reference.product_category",
    "landing.customer_consent",
    "reference.privacy_policy",
    "payments.transaction",
    "risk.chargebacks",
    "risk.device_trust",
    "landing.web_events",
    "marketing.campaigns",
    "erp.invoice",
    "erp.credit_note",
    "erp.purchase_orders",
    "erp.suppliers",
    "crm.support_cases",
    "reference.sla_policy"
]

print("\n========== ROW-COUNT VALIDATION ==========")

for table_name in tables:
    count = spark.table(f"{CAT}.{table_name}").count()
    print(f"{CAT}.{table_name}: {count} rows")
    assert count > 0, f"Validation failed: {CAT}.{table_name} has no rows"

print("\n========== SCHEMA VALIDATION ==========")

for table_name in tables:
    print(f"\n--- {CAT}.{table_name} ---")
    spark.table(f"{CAT}.{table_name}").printSchema()

print("\n========== FILE SOURCE VALIDATION ==========")

orders_check = spark.read.option("header", True).option("inferSchema", True).csv(orders_path)
print(f"Orders CSV rows: {orders_check.count()}")
orders_check.printSchema()

assert orders_check.count() > 0, "Orders CSV has no rows"

print("\nP1 source seeding and basic data checks completed.")
print("Mapping transformation correctness is NOT validated by this cell.")
print("Keep migration_pipeline dry_run=True for the next run.")